# Install Required Libraries

In [1]:
!pip install transformers
!pip install torch

# Import Required Libraries

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load Pretrained Chatbot Model

In [5]:
model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

# Create Response Generator Function

In [7]:
chat_history_ids = None

def get_response(user_input, chat_history_ids):

    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors='pt'
    )

    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)

    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    return response, chat_history_ids

# Build Interactive Chat Loop

In [9]:
print("Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.")

while True:

    user_input = input("You: ")

    if user_input.lower() == "exit" or user_input.lower() == "quit":
        print("Chatbot: Goodbye!")
        break

    response, chat_history_ids = get_response(user_input, chat_history_ids)

    print("Chatbot:", response)

Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.


You:  hello


Chatbot: Hello! :D


You:  what is AI


Chatbot: Artificial Intelligence


You:  explain me about AI


Chatbot: It's a computer program that makes you smarter.


You:  exit


Chatbot: Goodbye!
